In [1]:
pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [2]:
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    r=16,                          # Rank: the dimensionality of the low-rank update
    lora_alpha=32,                 # Alpha: scaling factor (2 * r)
    target_modules="all-linear",   # Apply to all linear layers in Gemma
    exclude_modules=".*(vision|audio).*", # Only use Text Layers
    lora_dropout=0.05,             # Light dropout for our small dataset
    bias="none",                   # Don't train bias terms
    task_type=TaskType.CAUSAL_LM,  # We're fine-tuning a causal language model
)

In [5]:
from transformers import AutoModelForCausalLM
from peft import get_peft_model

# Load the base model
base_model = AutoModelForCausalLM.from_pretrained("google/gemma-4-E4B-it")

# Wrap it with LoRA
model = get_peft_model(base_model, lora_config)

# Print the parameter summary
model.print_trainable_parameters()

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

trainable params: 38,879,232 || all params: 7,979,980,064 || trainable%: 0.4872


QLORA

In [2]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 35.3 MB/s eta 0:00:00


In [3]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # Load model weights in 4-bit
    bnb_4bit_quant_type="nf4",             # Use NormalFloat4
    bnb_4bit_compute_dtype=torch.bfloat16, # Compute in bfloat16
    bnb_4bit_use_double_quant=True,        # Enable double quantization
)

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "google/gemma-4-E4B-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

In [6]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# Prepare the quantized model for training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    exclude_modules=".*(vision|audio).*", # Only use Text Layers
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 54,734,848 || all params: 7,995,835,680 || trainable%: 0.6845


In [7]:
from peft import LoraConfig

dora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    use_dora=True,                  # Enable weight-decomposed adaptation
)